# cancer-recon-apoptosis — RUNG 1: real single-cell apoptosis kinetics (EARM)

Replaces the abstract ABM's instantaneous death with the **real, single-cell-calibrated** extrinsic-apoptosis cascade (PySB + EARM 1.0, Albeck/Sorger 2008). Shows: an all-or-none death-commitment **threshold**, a snap-action switch with hours-scale variable delay, cell-to-cell timing variability, and that **anti-apoptotic priming (Bcl-2/XIAP) raises the threshold** — the molecular basis of the Step-3 'selectivity = priming, not expression' finding.

**Ceiling (honest):** EARM takes *activated caspase-8 as input*. It does **not** prove a DR5 binder fires caspase-8 — the agonism crux is a wet-lab experiment (see `EVIDENCE_AND_HANDOFF.md`). Input knobs are proxies.

**Runtime:** CPU, no GPU. ~minutes. Validated locally before commit.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO); !git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — start run log + install PySB/BioNetGen
import sys
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('rung1_earm', repo_dir='.')
import importlib.util
if importlib.util.find_spec('pysb') is None:
    run_logged([sys.executable,'-m','pip','install','-q','pysb','bionetgen','cython'], RUNLOG, label='pip install pysb bionetgen cython')
print('[CELL 2] ✓')

In [ ]:
#@title Cell 3 — run RUNG 1 (EARM kinetics)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('rung1_earm', repo_dir='.')
rc = run_logged([sys.executable, '-u', 'scripts/11_earm_kinetics.py'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓' if rc==0 else '✗ (calibration gate)')
from IPython.display import Image, display
import os
if os.path.exists('runs/earm_kinetics/earm_kinetics.png'):
    display(Image('runs/earm_kinetics/earm_kinetics.png'))

In [ ]:
#@title Cell 4 — finalize + download run log
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('rung1_earm', repo_dir='.')
finalize(RUNLOG)

## Next rungs (the deep in-silico program)
RUNG 2 — clustering-geometry agonism predictor (calibrated on the DR5 valency ladder). RUNG 3 — PhysiCell tissue with real physics (feed Rung-1 death-latency in). RUNG 4 — patient priming-susceptibility map (DepMap/TCGA/CELLxGENE). All gate behind the wet-lab DR5-clustering/caspase-8 assay (`EVIDENCE_AND_HANDOFF.md`).